[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alsymiya/Argentina/blob/main/4.%20drawing_to_mechanism%20%28Colab%29.ipynb)

Open in Colab and run the Setup cell. The Setup cell pulls the helper `.py` files from GitHub automatically.

# Drawing -> MotionGen retrieval -> Differentiable refinement

Draw a curve. The MotionGen path-synthesis server returns its top-k four-bar guesses. The Argentina differentiable simulator then refines the best guess so its actual path-node trace matches what you drew.

This notebook acts as a *fake MotionGen frontend*: it builds the same `{knn, path, types}` payload the real MotionGen UI sends, parses the BSI-format response, and hands every RRRR candidate to the local simulator/optimizer for refinement.

## Setup

Run the next cell once. On Colab it locates the Argentina helper files in your Google Drive. The freehand canvas uses a plain HTML5 `<canvas>` (no `ipympl`, no kernel restart needed).

In [ ]:
# Pulls the Argentina helper .py files from GitHub if they're not already in
# the working directory. Falls back to mounting Google Drive if GitHub is
# unreachable. Run once at the top of the notebook.
import os, sys, urllib.request, urllib.parse

IN_COLAB = 'google.colab' in sys.modules
HELPERS = ['simulator.py', 'optimizer.py', 'visualizer.py',
           'motiongen_export.py', 'normalize.py', 'bsi_converter.py',
           'example_config.txt']
REPO_BASE = 'https://raw.githubusercontent.com/alsymiya/Argentina/main/'

if any(os.path.isfile(h) for h in HELPERS):
    print('helpers already in', os.getcwd())
else:
    n_ok = 0
    for h in HELPERS:
        try:
            urllib.request.urlretrieve(REPO_BASE + urllib.parse.quote(h), h)
            n_ok += 1
        except Exception:
            pass
    if any(os.path.isfile(h) for h in HELPERS):
        print(f'downloaded {n_ok}/{len(HELPERS)} helpers from GitHub')
    elif IN_COLAB:
        from google.colab import drive
        if not os.path.exists('/content/drive/MyDrive'):
            print('Mounting Google Drive (auth prompt) ...')
            drive.mount('/content/drive')
        for c in ['/content/drive/MyDrive/Argentina',
                  '/content/drive/My Drive/Argentina']:
            if any(os.path.isfile(os.path.join(c, h)) for h in HELPERS):
                os.chdir(c)
                if c not in sys.path:
                    sys.path.insert(0, c)
                print(f'using helpers from {c}')
                break
        else:
            print('WARNING: helpers not found via GitHub or Drive.')
    else:
        print('WARNING: helpers not found.  Drop .py files next to this notebook.')

## Configuration

In [ ]:
API_URL = 'https://motiongen-imgpathsyn-25dim-468-selectable-pnx6bopexa-uk.a.run.app/image-based-path-synthesis'
KNN     = 5             # how many candidate four-bars to fetch
TYPES   = ['RRRR']      # restrict to rotary four-bar RRRR
TIMEOUT = 30            # seconds

OPT_METRIC         = ['soft_chamfer', 'hausdorff']
OPT_METRIC_WEIGHTS = [1.0, 0.5]
OPT_LR             = 0.1
OPT_N_OUTER        = 60
OPT_TAU_ANNEAL     = True

print('API_URL =', API_URL)

In [ ]:
%matplotlib inline
import json, time, sys
from io import BytesIO
import numpy as np
import matplotlib.pyplot as plt
import requests
from IPython.display import HTML, display

from normalize        import resample_polyline
from bsi_converter    import convert_response, is_rrrr, actuator_is_rotary
from visualizer       import simulate
from optimizer        import fit_to_expected_path
from motiongen_export import build_motiongen_json, copy_to_clipboard

_state = {'points': [], 'points_px': [], 'response': None, 'candidates': []}

# --- HTML5 freehand canvas (Colab-native; no ipympl) --------------------
CANVAS_SIZE = 500
CANVAS_XLIM = (-5.0, 5.0)
CANVAS_YLIM = (-5.0, 5.0)

def _px_to_data(points_px):
    pts = np.asarray(points_px, dtype=float)
    if pts.size == 0:
        return pts.reshape(0, 2)
    x = pts[:, 0] / CANVAS_SIZE * (CANVAS_XLIM[1] - CANVAS_XLIM[0]) + CANVAS_XLIM[0]
    y = CANVAS_YLIM[1] - pts[:, 1] / CANVAS_SIZE * (CANVAS_YLIM[1] - CANVAS_YLIM[0])
    return np.column_stack([x, y])

_CANVAS_HTML = """
<div>
  <canvas id="drawcanvas" width="500" height="500"
          style="border:1px solid #888; cursor:crosshair; touch-action:none;"></canvas>
  <div style="margin-top:6px; font-family:monospace;">
    <button id="clearbtn">Clear</button>
    <span id="status" style="margin-left:10px;">0 points</span>
  </div>
</div>
<script>
(function() {
  const canvas = document.getElementById('drawcanvas');
  const ctx = canvas.getContext('2d');
  const status = document.getElementById('status');
  let drawing = false, points = [];
  function grid() {
    ctx.clearRect(0,0,canvas.width,canvas.height);
    ctx.strokeStyle='#eee'; ctx.lineWidth=1;
    for (let i=0;i<=10;i++){let p=i/10*canvas.width;
      ctx.beginPath();ctx.moveTo(p,0);ctx.lineTo(p,canvas.height);ctx.stroke();
      ctx.beginPath();ctx.moveTo(0,p);ctx.lineTo(canvas.width,p);ctx.stroke();}
    ctx.strokeStyle='#bbb';
    ctx.beginPath();ctx.moveTo(canvas.width/2,0);ctx.lineTo(canvas.width/2,canvas.height);ctx.stroke();
    ctx.beginPath();ctx.moveTo(0,canvas.height/2);ctx.lineTo(canvas.width,canvas.height/2);ctx.stroke();
  }
  grid();
  function pos(e){const r=canvas.getBoundingClientRect();return [e.clientX-r.left, e.clientY-r.top];}
  function send(){if(window.google&&google.colab&&google.colab.kernel){
      google.colab.kernel.invokeFunction('notebook.set_points',[points],{});}}
  canvas.addEventListener('mousedown',e=>{drawing=true;points=[pos(e)];grid();
    ctx.strokeStyle='#3366cc';ctx.lineWidth=2;ctx.beginPath();ctx.moveTo(points[0][0],points[0][1]);});
  canvas.addEventListener('mousemove',e=>{if(!drawing)return;const p=pos(e);points.push(p);
    ctx.lineTo(p[0],p[1]);ctx.stroke();status.textContent=points.length+' points';});
  function finish(){if(!drawing)return;drawing=false;
    status.textContent=points.length+' points (sent to Python)';send();}
  canvas.addEventListener('mouseup',finish);
  canvas.addEventListener('mouseleave',finish);
  document.getElementById('clearbtn').addEventListener('click',()=>{
    points=[];grid();status.textContent='0 points';send();});
})();
</script>
"""

def _register_canvas_callback():
    if not IN_COLAB:
        print('Note: the HTML5 canvas callback only works on Colab.')
        print('On local Jupyter, set _state["points"] = [[x,y], ...] manually.')
        return
    from google.colab import output as _o
    def _cb(points_px):
        _state['points_px'] = list(points_px)
        _state['points'] = _px_to_data(points_px).tolist()
    _o.register_callback('notebook.set_points', _cb)

def show_draw_canvas():
    _state['points'] = []
    _state['points_px'] = []
    _register_canvas_callback()
    display(HTML(_CANVAS_HTML))
    print('Draw in the canvas above (click + drag, release to finish), then run the next cell.')

# --- assembly-mode / wrap-around branch picker --------------------------
def branch_near(trace, anchor):
    """Contiguous block of finite trace samples including the sample closest to
    `anchor`. Handles cyclic wrap AND picks one assembly-mode branch."""
    trace  = np.asarray(trace, dtype=float)
    anchor = np.asarray(anchor, dtype=float).reshape(2)
    T = trace.shape[0]
    finite = np.isfinite(trace).all(axis=1)
    if not finite.any():
        return trace[:0]
    d = np.full(T, np.inf)
    d[finite] = np.linalg.norm(trace[finite] - anchor, axis=1)
    i0 = int(np.argmin(d))
    idx = [i0]
    j = (i0 + 1) % T
    while j != i0 and finite[j]:
        idx.append(j); j = (j + 1) % T
    j = (i0 - 1) % T
    while j != i0 and finite[j] and j not in idx:
        idx.insert(0, j); j = (j - 1) % T
    return trace[idx]

# --- save figure + (Windows) copy image to clipboard --------------------
def save_and_copy_image(fig, basename='merged_refined'):
    ts = time.strftime('%Y%m%d-%H%M%S')
    png_path = f'{basename}_{ts}.png'
    fig.savefig(png_path, dpi=150, bbox_inches='tight', facecolor='white')
    if not sys.platform.startswith('win'):
        return png_path, False
    try:
        from PIL import Image
        import ctypes
        from ctypes import wintypes
        img = Image.open(png_path).convert('RGB')
        buf = BytesIO(); img.save(buf, 'BMP')
        data = buf.getvalue()[14:]
        user32   = ctypes.WinDLL('user32',   use_last_error=True)
        kernel32 = ctypes.WinDLL('kernel32', use_last_error=True)
        CF_DIB = 8; GMEM_MOVEABLE = 0x0002
        kernel32.GlobalAlloc.argtypes = [wintypes.UINT, ctypes.c_size_t]
        kernel32.GlobalAlloc.restype  = wintypes.HGLOBAL
        kernel32.GlobalLock.argtypes  = [wintypes.HGLOBAL]
        kernel32.GlobalLock.restype   = ctypes.c_void_p
        user32.SetClipboardData.argtypes = [wintypes.UINT, wintypes.HANDLE]
        user32.SetClipboardData.restype  = wintypes.HANDLE
        if not user32.OpenClipboard(0):
            return png_path, False
        try:
            user32.EmptyClipboard()
            h_mem = kernel32.GlobalAlloc(GMEM_MOVEABLE, len(data))
            if not h_mem: return png_path, False
            ptr = kernel32.GlobalLock(h_mem)
            if not ptr: return png_path, False
            ctypes.memmove(ptr, data, len(data))
            kernel32.GlobalUnlock(h_mem)
            user32.SetClipboardData(CF_DIB, h_mem)
            return png_path, True
        finally:
            user32.CloseClipboard()
    except Exception as e:
        print(f'  image clipboard copy failed: {e}')
        return png_path, False


## Step 1 - Draw a curve

Click and drag to draw; release to finish. Hit **Clear** to start over. The stroke is sent to Python automatically on release.

In [ ]:
show_draw_canvas()

## Step 2 - Send the curve to MotionGen

Builds `{"knn": KNN, "path": [[x, y], ...], "types": ['RRRR']}` and POSTs to the synthesis endpoint. The response is `{"version": ..., "solutions": [{B, S, I, p, c, error}, ...]}`.

In [ ]:
drawn = np.array(_state['points'], dtype=float)
assert drawn.shape[0] >= 20, 'Please draw a longer stroke (>=20 points).'

path_payload = drawn if len(drawn) <= 200 else resample_polyline(drawn, 200)

payload = {'knn': KNN, 'path': path_payload.tolist(), 'types': TYPES}
print('POST', API_URL)
print(f'  payload: knn={KNN}, path_len={len(path_payload)}, types={TYPES}')

t0 = time.time()
r = requests.post(API_URL, json=payload, timeout=TIMEOUT)
dt = time.time() - t0
r.raise_for_status()
response = r.json()
_state['response'] = response
print(f'  response in {dt*1000:.0f} ms; version={response.get("version")!r}; solutions={len(response.get("solutions", []))}')

if response.get('solutions'):
    s = response['solutions'][0]
    print()
    print('First candidate:')
    print(f'  is_rrrr={is_rrrr(s)}  rotary={actuator_is_rotary(s)}  error={s.get("error"):.4f}')
    print(f'  B has {len(s["B"])} bodies x {len(s["B"][0])} joints; c={s["c"]}; I={s["I"]}')
    print(f'  p[0..2] = {s["p"][:3]}')

## Step 3 - Simulate each candidate, overlay on the drawing

Each trace is filtered through `branch_near(...)` anchored at the candidate's initial path-node position.

In [ ]:
candidates = convert_response(_state['response'])
print(f'kept {len(candidates)} RRRR candidate(s)')
_state['candidates'] = candidates

if not candidates:
    raise RuntimeError('Server returned no usable RRRR candidates. Try drawing a different curve.')

colors = ['#cc3333', '#e07b00', '#22aa22', '#9b59b6', '#3399cc', '#666666']

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(drawn[:, 0], drawn[:, 1], '-', color='black', lw=2.5, label='drawn', zorder=10)

for i, cfg in enumerate(candidates):
    try:
        _, trace = simulate(cfg['JJ'], cfg['PSlice'], cfg['motor'],
                            cfg['fixed_nodes'], cfg['path_node'], n_theta=200)
        anchor = np.asarray(cfg['PSlice'])[cfg['path_node']]
        arc = branch_near(trace, anchor)
        col = colors[i % len(colors)]
        if len(arc) > 0:
            ax.plot(arc[:, 0], arc[:, 1], '-', color=col, lw=1.5, alpha=0.85,
                    label=f'[{i}] err={cfg["error"]:.3f}')
        else:
            print(f'  candidate {i}: no reachable samples')
        ps = np.asarray(cfg['PSlice'])
        for j in cfg['fixed_nodes']:
            ax.plot(ps[j, 0], ps[j, 1], 's', color=col, ms=6)
    except Exception as e:
        print(f'  candidate {i} failed to simulate: {e}')

ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.legend(loc='upper right', fontsize=9)
ax.set_title('drawn curve (black) + each candidate trace (anchored branch)')
plt.tight_layout(); plt.show()

## Step 4 - Pick a candidate to refine

In [ ]:
pick_idx = 0
chosen = candidates[pick_idx]
print(f'picked candidate {pick_idx}  error={chosen["error"]:.4f}')
print(f'  fixed_nodes={chosen["fixed_nodes"]}  motor={chosen["motor"]}  path_node={chosen["path_node"]}')

_, trace0_raw = simulate(chosen['JJ'], chosen['PSlice'], chosen['motor'],
                         chosen['fixed_nodes'], chosen['path_node'], n_theta=200)
anchor0 = np.asarray(chosen['PSlice'])[chosen['path_node']]
trace0 = branch_near(trace0_raw, anchor0)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(drawn[:, 0], drawn[:, 1], '-', color='black', lw=2.5, label='drawn')
if len(trace0) > 0:
    ax.plot(trace0[:, 0], trace0[:, 1], '-', color='#cc3333', lw=1.8, label='candidate trace')
ps = np.asarray(chosen['PSlice'])
ax.plot(ps[:, 0], ps[:, 1], 'o', color='#3366cc', ms=6, label='joints')
for j in chosen['fixed_nodes']:
    ax.plot(ps[j, 0], ps[j, 1], 's', color='black', ms=8)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend(loc='upper right')
ax.set_title(f'candidate {pick_idx} BEFORE refinement')
plt.show()

## Step 5 - Differentiable refinement

L-BFGS through the differentiable simulator. The BEFORE/AFTER figure is saved as a PNG and (on Windows) copied to the clipboard so you can paste it into MotionGen alongside the JSON.

In [ ]:
target = drawn

t0 = time.time()
result = fit_to_expected_path(
    JJ             = chosen['JJ'],
    PSlice         = chosen['PSlice'],
    expected_path  = target,
    motor          = chosen['motor'],
    fixed_nodes    = chosen['fixed_nodes'],
    path_node      = chosen['path_node'],
    method         = 'lbfgs',
    metric         = OPT_METRIC,
    metric_weights = OPT_METRIC_WEIGHTS,
    n_outer        = OPT_N_OUTER,
    lr             = OPT_LR,
    tau_anneal     = OPT_TAU_ANNEAL,
    verbose        = True,
)
dt = time.time() - t0
print(f'refinement done in {dt:.2f}s')

PSlice_opt = np.asarray(result['x_optimized']).reshape(-1, 2)

_, trace1_raw = simulate(chosen['JJ'], PSlice_opt, chosen['motor'],
                         chosen['fixed_nodes'], chosen['path_node'], n_theta=200)
anchor1 = PSlice_opt[chosen['path_node']]
trace1 = branch_near(trace1_raw, anchor1)

fig5, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, ttl, tr, ps in [
    (axes[0], 'BEFORE - candidate trace', trace0, np.asarray(chosen['PSlice'])),
    (axes[1], 'AFTER - refined trace',    trace1, PSlice_opt),
]:
    ax.plot(drawn[:, 0], drawn[:, 1], '-', color='black', lw=2.5, label='drawn')
    if len(tr) > 0:
        ax.plot(tr[:, 0], tr[:, 1], '-', color='#cc3333', lw=1.8, label='trace')
    ax.plot(ps[:, 0], ps[:, 1], 'o', color='#3366cc', ms=6)
    for j in chosen['fixed_nodes']:
        ax.plot(ps[j, 0], ps[j, 1], 's', color='black', ms=8)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend(loc='upper right'); ax.set_title(ttl)
plt.tight_layout(); plt.show()

img_path, img_clip = save_and_copy_image(fig5, basename='merged_refined')
print(f'image saved: {img_path}')
print(f'image clipboard: {"OK - paste into MotionGen" if img_clip else "unavailable on this OS; paste the PNG file manually"}')

## Step 6 - Export the mechanism to MotionGen

`build_motiongen_json` returns a **dict** - we `json.dumps` it before writing / copying / printing.

In [ ]:
refined = {
    'JJ'         : chosen['JJ'],
    'PSlice'     : PSlice_opt,
    'motor'      : chosen['motor'],
    'fixed_nodes': chosen['fixed_nodes'],
    'path_node'  : chosen['path_node'],
}
mg_dict = build_motiongen_json(refined['JJ'], refined['PSlice'],
                                refined['motor'], refined['fixed_nodes'],
                                refined['path_node'])
mg_text = json.dumps(mg_dict)

ts = time.strftime('%Y%m%d-%H%M%S')
out_path = f'merged_refined_{ts}.txt'
with open(out_path, 'w', encoding='utf-8') as f:
    f.write(mg_text)

copied = copy_to_clipboard(mg_text)
print(f'wrote {out_path}; clipboard: {"OK" if copied else "unavailable (paste from the file)"}')
print()
#print('--- MotionGen JSON (first 400 chars) ---')
#print(mg_text[:400] + ('...' if len(mg_text) > 400 else ''))
print('--- MotionGen JSON ---')
print(mg_text)